# 2. Fitting a model to photometry

Fit an `artefact` model to the photometry of HD 104860 (`example/HD104860_photometry.txt`, obtained from Vizier with PySED):

1. read the photometry and scale the stellar photosphere to the optical and near-IR points,
2. fit the dust mass, size distribution slope and belt radius to the infrared excess with `lmfit`,
3. inspect the best fit, residuals and uncertainties,
4. optionally sample the posterior with `emcee`.

In [ ]:
import sys
from pathlib import Path

# The notebooks live in notebooks/, the helpers in example/
sys.path.insert(0, str(Path.cwd().parent / "example"))

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

In [ ]:
import lmfit
from example_utils import EXAMPLE, base_parameters, model_at, plot_sed, run_model

## Photometry

The file columns are wavelength (um), flux density (mJy), uncertainty (mJy) and a reference label.

In [ ]:
wave, flux, unc = np.loadtxt(
    EXAMPLE / "HD104860_photometry.txt", delimiter=",", comments="#",
    usecols=(0, 1, 2), unpack=True,
)
photometry = (wave, flux, unc)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(wave, flux, unc, fmt="o", mfc="white", mec="k", ecolor="k")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Wavelength (um)")
ax.set_ylabel("Flux density (mJy)")
plt.show()

## The model

The star is a G2V template rescaled to the photometry between 0.45 and 5 um by `scale_star`, with the stellar luminosity, radius and distance of HD 104860. The belt is a Gaussian ring whose peak radius is fitted; its width is held fixed.

The fit uses the quick `mode="bb"` grain temperatures. The final model is re-run with `mode="full"`.

In [ ]:
SCALE_RANGE = (0.45, 5.0)   # um, points used to scale the photosphere
EXCESS_MIN = 10.0           # um, points used to fit the disc


def parameters_for(theta):
    return base_parameters(
        lstar=1.318, rstar=0.94, tstar=5950.0, dstar=48.0,
        ngrain=31, nwav=120, amin=4.3,
        dtype="gauss", nring=21, rfwhm=60.0,
        mdust=10 ** theta["log_mdust"],
        q=theta["q"],
        rpeak=theta["rpeak"],
    )


excess = wave > EXCESS_MIN


def residual(theta):
    model = run_model(
        parameters_for(theta), photometry=photometry,
        scale_wave=SCALE_RANGE, emission_mode="bb",
    )
    return (model_at(model, wave[excess]) - flux[excess]) / unc[excess]

## Fit

Free parameters are $\log_{10} M_\mathrm{dust}$ (Earth masses), the size distribution slope $q$ and the belt peak radius. This takes a few minutes.

In [ ]:
params = lmfit.Parameters()
params.add("log_mdust", value=-2.0, min=-5.0, max=0.0)
params.add("q", value=-3.5, min=-4.5, max=-3.0)
params.add("rpeak", value=110.0, min=20.0, max=250.0)

result = lmfit.minimize(residual, params, method="least_squares")
print(lmfit.fit_report(result))

In [ ]:
best = run_model(
    parameters_for(result.params.valuesdict()), photometry=photometry,
    scale_wave=SCALE_RANGE, emission_mode="full",
)

fig, (ax, ax_res) = plt.subplots(
    2, 1, figsize=(7, 7), sharex=True, gridspec_kw=dict(height_ratios=[3, 1])
)
plot_sed(best, ax=ax, title="HD 104860 best fit")
ax.set_xlabel("")
ax_res.axhline(0, color="k", lw=0.5)
ax_res.plot(wave[excess], residual(result.params), "ko")
ax_res.set_xscale("log")
ax_res.set_xlabel("Wavelength (um)")
ax_res.set_ylabel("(model - data) / sigma")
plt.show()

The `chisqr`, `redchi` and parameter uncertainties from `lmfit` assume a locally linear, Gaussian problem. Fitting `q` and `mdust` together is degenerate, so check the correlation block in the report above and consider the sampling below.

## Posterior sampling (optional)

Requires `pip install emcee corner`. Each step runs the full model once per walker, so this is slow; reduce `nwalkers` and `steps` for a quick look.

In [ ]:
RUN_MCMC = False

if RUN_MCMC:
    import corner

    fit = lmfit.minimize(
        residual, result.params, method="emcee",
        nwalkers=24, steps=600, burn=200, thin=5, progress=True,
    )
    corner.corner(fit.flatchain, labels=list(fit.var_names), quantiles=[0.16, 0.5, 0.84])
    plt.show()

## Derived quantities

In [ ]:
print(f"Dust mass : {10 ** result.params['log_mdust'].value:.3g} M_earth")
print(f"Belt mass : {best.calculate_disc_mass():.3g} M_earth")